# Step 2: Train with a live dashboard

1. Check the data and the chat format
2. Start training in the background
3. Watch a live loss chart. It **stops by itself** when val loss stops improving
4. Pick the best checkpoint (not just the last one)
5. Read samples from different checkpoints

Run `01_prepare_dataset.ipynb` first.

## Settings

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
DATA_DIR = "data"

ITERS = 1500            # about 2 passes over 2850 plots at batch size 4
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
NUM_LAYERS = 16
MAX_SEQ_LEN = 768       # same as 01_prepare_dataset.ipynb

EVAL_EVERY = 100        # val loss + a saved checkpoint every this many steps
PATIENCE = 3            # stop if val loss has not improved for this many checks

ADAPTER_PATH = "adapters"
BEST_PATH = "adapters_best"
LOG_FILE = "train.log"

## 1. Check the data and the format

In [ ]:
import json, os, re, shutil, subprocess, sys, time

n_train = sum(1 for _ in open(f"{DATA_DIR}/train.jsonl"))
n_valid = sum(1 for _ in open(f"{DATA_DIR}/valid.jsonl"))
steps_per_epoch = n_train / BATCH_SIZE
print(f"train: {n_train}  valid: {n_valid}")
print(f"One pass over the data = {steps_per_epoch:.0f} steps. ITERS={ITERS} is about {ITERS / steps_per_epoch:.1f} passes.")

This prints the exact text the model sees during training. You should see the model's own chat tokens (for Qwen: `<|im_start|>` and `<|im_end|>`). The trainer builds this with the model's built-in chat template, the same one `compare.py` uses.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
ex = json.loads(open(f"{DATA_DIR}/train.jsonl").readline())
print(tok.apply_chat_template(ex["messages"], tokenize=False)[:1200])

## 2. Dashboard helpers

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import clear_output

def parse_log(path=LOG_FILE):
    train, val, speed, mem = [], [], [], None
    if not os.path.exists(path):
        return train, val, speed, mem
    for line in open(path):
        m = re.search(r"Iter (\d+): Train loss ([\d.]+)", line)
        if m:
            it = int(m.group(1))
            train.append((it, float(m.group(2))))
            s = re.search(r"Tokens/sec ([\d.]+)", line)
            if s:
                speed.append((it, float(s.group(1))))
            p = re.search(r"Peak mem ([\d.]+)", line)
            if p:
                mem = float(p.group(1))
        m = re.search(r"Iter (\d+): Val loss ([\d.]+)", line)
        if m:
            val.append((int(m.group(1)), float(m.group(2))))
    return train, val, speed, mem

def best_val(val):
    cand = [(l, i) for i, l in val if i > 1]      # step 1 is the "before training" score
    return min(cand) if cand else (None, None)

def should_stop(val, patience):
    cand = [(i, l) for i, l in val if i > 1]
    if len(cand) <= patience:
        return False
    _, best_i = best_val(val)
    return sum(1 for i, _ in cand if i > best_i) >= patience

def draw(train, val, speed, mem):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    if train:
        its, ls = zip(*train)
        ax[0].plot(its, ls, alpha=0.25, color="tab:blue", label="train loss (raw)")
        ax[0].plot(its, pd.Series(ls).rolling(5, min_periods=1).mean(), color="tab:blue", label="train loss (smoothed)")
    if val:
        vi, vl = zip(*val)
        ax[0].plot(vi, vl, "o-", color="tab:orange", label="val loss")
        bl, bi = best_val(val)
        if bi:
            ax[0].axvline(bi, ls="--", color="gray")
            ax[0].set_title(f"Best val loss {bl:.3f} at step {bi}")
    ax[0].set_xlabel("step"); ax[0].set_ylabel("loss"); ax[0].legend()
    if speed:
        si, sv = zip(*speed)
        ax[1].plot(si, sv, color="tab:green")
    ax[1].set_xlabel("step"); ax[1].set_ylabel("tokens / sec")
    ax[1].set_title(f"Speed (peak memory {mem} GB)" if mem else "Speed")
    plt.tight_layout(); plt.show()

## 3. Start training (runs in the background)

Training writes to `train.log`. Do not restart the kernel while it runs.

In [ ]:
cmd = [
    sys.executable, "-m", "mlx_lm.lora",
    "--model", BASE_MODEL,
    "--train",
    "--data", DATA_DIR,
    "--iters", str(ITERS),
    "--batch-size", str(BATCH_SIZE),
    "--learning-rate", str(LEARNING_RATE),
    "--num-layers", str(NUM_LAYERS),
    "--max-seq-length", str(MAX_SEQ_LEN),
    "--mask-prompt",
    "--grad-checkpoint",
    "--steps-per-eval", str(EVAL_EVERY),
    "--save-every", str(EVAL_EVERY),
    "--adapter-path", ADAPTER_PATH,
]
print(" ".join(cmd))

proc = subprocess.Popen(
    cmd,
    stdout=open(LOG_FILE, "w"),
    stderr=subprocess.STDOUT,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)
print("Started. Run the next cell to watch it.")

## 4. Live dashboard

Refreshes every 20 seconds and stops training early if val loss has not improved for `PATIENCE` checks.
Pressing the stop button on this cell only stops the **chart**. Training keeps going, and you can re-run the cell to watch again.

In [ ]:
try:
    while True:
        train, val, speed, mem = parse_log()
        clear_output(wait=True)
        draw(train, val, speed, mem)
        if train:
            print(f"step {train[-1][0]} / {ITERS}   latest train loss {train[-1][1]:.3f}")
        if proc.poll() is not None:
            print("Training finished.")
            break
        if should_stop(val, PATIENCE):
            proc.terminate(); proc.wait()
            print(f"Val loss did not improve for {PATIENCE} checks in a row. Stopped early.")
            break
        time.sleep(20)
except KeyboardInterrupt:
    print("Chart stopped. Training is still running in the background.")

## 5. Pick the best checkpoint

Training may have gone past the best point, so we copy the checkpoint with the lowest val loss into `adapters_best`.

In [ ]:
train, val, speed, mem = parse_log()
best_l, best_i = best_val(val)
print(f"Before training, val loss was {val[0][1]:.3f}" if val and val[0][0] == 1 else "")
print(f"Best val loss {best_l:.3f} at step {best_i}")

src = f"{ADAPTER_PATH}/{best_i:07d}_adapters.safetensors"
if not os.path.exists(src):
    src = f"{ADAPTER_PATH}/adapters.safetensors"   # fall back to the final save
    print("Numbered checkpoint not found, using", src)

os.makedirs(BEST_PATH, exist_ok=True)
shutil.copy(src, f"{BEST_PATH}/adapters.safetensors")
shutil.copy(f"{ADAPTER_PATH}/adapter_config.json", f"{BEST_PATH}/adapter_config.json")
print("Saved to", BEST_PATH)

## 6. Read samples from different checkpoints

The lowest val loss is a good guess for "best", but it does not always mean the best writing.
This shows the same prompts at the first, best and last checkpoint, so you can compare with your own eyes.

In [ ]:
import gc, tempfile
from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler

PROMPTS = [
    "Genre: sci-fi. Keywords: haunted spaceship, betrayal",
    "Genre: comedy. Keywords: samosa shop, rivalry, wedding",
]

def sample_checkpoint(step, max_tokens=300):
    tmp = tempfile.mkdtemp()
    shutil.copy(f"{ADAPTER_PATH}/{step:07d}_adapters.safetensors", f"{tmp}/adapters.safetensors")
    shutil.copy(f"{ADAPTER_PATH}/adapter_config.json", f"{tmp}/adapter_config.json")
    model, tokenizer = load(BASE_MODEL, adapter_path=tmp)
    sampler = make_sampler(temp=0.8, top_p=0.9)
    for p in PROMPTS:
        text = tokenizer.apply_chat_template(
            [{"role": "user", "content": p}], add_generation_prompt=True, tokenize=False
        )
        print(f"### step {step} | {p}\n")
        print(generate(model, tokenizer, prompt=text, max_tokens=max_tokens, sampler=sampler))
        print("\n" + "=" * 80 + "\n")
    del model
    gc.collect()

steps = sorted(
    int(m.group(1)) for f in os.listdir(ADAPTER_PATH)
    if (m := re.match(r"(\d+)_adapters\.safetensors", f))
)
for s in sorted({steps[0], best_i, steps[-1]}):
    sample_checkpoint(s)

## Next

In `compare.py` and `publish.py`, change this line:

```python
ADAPTER_PATH = "adapters_best"
```